# Define the slow-resolution label

Build intake features and a binary slow-resolution label for all NYC `HEAT/HOT WATER` requests in the Jan 15–22 event window and Jan 24–30 quiet window of 2024. These are full window populations from Socrata `erm2-nwe9`.

The threshold is the quiet-window closed-ticket p90, computed from the pulled data. It is a project benchmark, not a City service standard. The event week had higher intake volume; temperature is not present, so no weather cause is claimed.

Features are intake-time fields only. Open tickets remain unlabeled. Precise geography and post-closure outcomes are excluded from features; dates and window labels are audit/split fields only. No Gemini or Qdrant calls are used.

In [1]:
import hashlib, json, time

import httpx
import numpy as np
import pandas as pd

from resolveiq.config import REPO_ROOT, get_settings

settings = get_settings()
url = f"{settings.nyc311_base_url}/{settings.nyc311_dataset_id}.json"
meta_url = url.replace("/resource/", "/api/views/").replace(".json", "")
out_dir = REPO_ROOT / "data" / "labelled"
out_dir.mkdir(parents=True, exist_ok=True)

print(f"repo     {REPO_ROOT}")
print(f"records  {url}")
print(f"metadata {meta_url}")
print(f"output   {out_dir}")

repo     C:\Users\mejdi\Documents\resolveiq
records  https://data.cityofnewyork.us/resource/erm2-nwe9.json
metadata https://data.cityofnewyork.us/api/views/erm2-nwe9
output   C:\Users\mejdi\Documents\resolveiq\data\labelled


## Which fields exist, and when the city writes them

Nothing in the rest of this notebook names a column from memory. The city's own metadata response is read first,
and the allow/deny lists below are built against it — the display label is not the identifier, `fieldName` is.

In [2]:
meta = httpx.get(meta_url, timeout=60.0)
meta.raise_for_status()
source_fields = pd.DataFrame(
    [(c["fieldName"], c.get("typeName", ""), (c.get("description") or "(source gives no description)").strip())
     for c in meta.json()["columns"]],
    columns=["fieldName", "typeName", "city_description"],
)
source_fields["city_description"] = source_fields["city_description"].str.slice(0, 68)
ALL_SOURCE = set(source_fields["fieldName"])
print(f"{len(source_fields)} fields   {meta.elapsed.total_seconds():.2f}s   no request body sent")
print(source_fields.to_string(index=False))

48 fields   0.79s   no request body sent
                     fieldName typeName                                                     city_description
                    unique_key              Unique identifier of a Service Request (SR) in the open data set
                  created_date                                                          Date SR  was created
                   closed_date                                       Date SR was closed by responding agency
                        agency                                  Acronym of responding City Government Agency
                   agency_name                         Full Agency name of responding City Government Agency
                complaint_type          This is the first level of a hierarchy identifying the topic of the 
                    descriptor          This is associated to the Problem (formerly Complaint Type), and pro
                  descriptor_2          A third level of detail about the Problem (form

## Intake-field policy

Only fields in the explicit allowlist and present in the Socrata schema may enter the feature table; all other source columns are denied. `city` and `incident_zip` are excluded, as are addresses, coordinates, stations, and intersections. `borough` is retained as coarse geography and flagged for fairness/proxy review.

In [3]:
APPROVED_INTAKE = ["complaint_type", "descriptor", "descriptor_2", "location_type",
                   "borough", "open_data_channel_type", "agency"]
AUDIT_ONLY = ["unique_key", "created_date", "window"]
REVIEW_FLAGS = {
    "borough": "coarse service jurisdiction — needs a fairness/proxy review before it is served as a feature",
}
NOTES = {
    "city": "redundant with borough at a different granularity (25 of 29 values are Queens neighbourhoods)",
    "incident_zip": "precise geography and a nominal identifier; excluded from v1 pending a privacy/fairness policy",
}

unknown = [f for f in APPROVED_INTAKE + AUDIT_ONLY if f not in ALL_SOURCE and f != "window"]
assert not unknown, f"approved but absent from the source, so it would return nothing: {unknown}"

INTAKE_FIELDS = [f for f in APPROVED_INTAKE if f in ALL_SOURCE]
FORBIDDEN_FIELDS = sorted(ALL_SOURCE - set(INTAKE_FIELDS) - {"unique_key", "created_date"})

assert set(INTAKE_FIELDS).isdisjoint(FORBIDDEN_FIELDS)
assert len(INTAKE_FIELDS) + len(FORBIDDEN_FIELDS) + 2 == len(ALL_SOURCE)
print(f"source fields {len(ALL_SOURCE)} = approved intake {len(INTAKE_FIELDS)} "
      f"+ audit 2 + forbidden {len(FORBIDDEN_FIELDS)}   (fail-closed: unlisted means forbidden)")
print(f"approved intake: {INTAKE_FIELDS}")
print(f"audit only     : {AUDIT_ONLY}")
print("forbidden by default:")
for i in range(0, len(FORBIDDEN_FIELDS), 4):
    print("   " + ", ".join(FORBIDDEN_FIELDS[i:i + 4]))
print("\nexcluded on purpose:")
for name, why in NOTES.items():
    if name in ALL_SOURCE:
        print(f"   {name:<14} {why}")
print(f"   {'borough':<14} kept, but {REVIEW_FLAGS['borough']}")

source fields 48 = approved intake 7 + audit 2 + forbidden 39   (fail-closed: unlisted means forbidden)
approved intake: ['complaint_type', 'descriptor', 'descriptor_2', 'location_type', 'borough', 'open_data_channel_type', 'agency']
audit only     : ['unique_key', 'created_date', 'window']
forbidden by default:
   :@computed_region_92fq_4b7q, :@computed_region_f5dn_yrer, :@computed_region_sbqj_enih, :@computed_region_yeji_bk3q
   address_type, agency_name, bbl, bridge_highway_direction
   bridge_highway_name, bridge_highway_segment, city, closed_date
   community_board, council_district, cross_street_1, cross_street_2
   due_date, facility_type, incident_address, incident_zip
   intersection_street_1, intersection_street_2, landmark, latitude
   location, longitude, park_borough, park_facility_name
   police_precinct, resolution_action_updated_date, resolution_description, road_ramp
   status, street_name, taxi_company_borough, taxi_pick_up_location
   vehicle_type, x_coordinate_state

## Pull both windows in full

Notebook 01's two window files carry only five columns and none of them are intake features, so this is a fresh
pull. Paging is `$limit 20000` + `$order unique_key`, a stable sort, until a short page arrives.

The `closed_date is not null` filter that notebook 01 used is **removed** on purpose: open tickets are real intake
rows and must appear in the features table. `closed_date` and `status` are selected only to compute and audit the
label, and are deleted from the feature table before anything is written.

In [4]:
WINDOWS = {"event_2024-01-15_22": ("2024-01-15", "2024-01-23"),
           "quiet_2024-01-24_30": ("2024-01-24", "2024-01-31")}
HEAT = "lower(complaint_type) = 'heat/hot water'"
SELECT = ", ".join(["unique_key", "created_date"] + INTAKE_FIELDS + ["closed_date", "status"])
WHERE_BY_WINDOW = {label: (f"{HEAT} and created_date >= '{d0}T00:00:00'"
                  f" and created_date < '{d1}T00:00:00'") for label, (d0, d1) in WINDOWS.items()}
PULL_PARAMS = {"$select": SELECT, "$order": "unique_key", "$limit": "20000"}


def fetch_all(where):
    rows, offset, pages = [], 0, 0
    while True:
        response = httpx.get(url, params={**PULL_PARAMS, "$where": where, "$offset": str(offset)},
                             timeout=180.0)
        if response.status_code >= 400:
            print(response.text[:400])
        response.raise_for_status()
        batch = response.json()
        rows.extend(batch)
        pages += 1
        if len(batch) < 20000:
            return rows, pages
        offset += 20000


requests_used = 1
started = time.perf_counter()
frames = {}
for label, where in WHERE_BY_WINDOW.items():
    rows, pages = fetch_all(where)
    requests_used += pages
    part = pd.DataFrame(rows)
    returned = set(part.columns)
    for col in INTAKE_FIELDS + ["closed_date", "status"]:
        if col not in part:
            part[col] = pd.NA
    part["created_date"] = pd.to_datetime(part["created_date"])
    part["window"] = label
    frames[label] = (part, returned)
    print(f"{label}  {len(rows):>6,} created   {pages} page(s)")
pull_s = time.perf_counter() - started

data = pd.concat([f for f, _ in frames.values()], ignore_index=True)
returned_cols = sorted(set().union(*[r for _, r in frames.values()]))
missing_cols = [c for c in INTAKE_FIELDS if c not in returned_cols]
print(f"\n{len(data):,} rows in {pull_s:.1f}s across {requests_used - 1} requests "
      f"({requests_used} including the metadata read)")
print(f"columns the API returned: {len(returned_cols)} of {len(SELECT.split(', '))} asked for"
      + (f"   never returned: {missing_cols}" if missing_cols else ""))

event_2024-01-15_22  22,811 created   2 page(s)
quiet_2024-01-24_30   7,315 created   1 page(s)

30,126 rows in 16.4s across 3 requests (4 including the metadata read)
columns the API returned: 11 of 11 asked for


## Integrity gate: is this pull what we think it is

Five checks, and the pass/fail table is printed before any label exists. A duplicate from `$offset` paging, a row
outside its own window, or a silently missing column would each quietly corrupt everything downstream, so none of
them is allowed to pass unnoticed.

In [6]:
def count(where):
    response = httpx.get(url, params={"$select": "count(*) as n", "$where": where}, timeout=60.0)
    response.raise_for_status()
    return int(response.json()[0]["n"])


api_counts = {label: count(where) for label, where in WHERE_BY_WINDOW.items()}
checks = []


def check(name, ok, detail):
    checks.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})


check("unique_key is unique", data["unique_key"].is_unique,
      f"{data['unique_key'].nunique():,} distinct of {len(data):,} rows")
check("created_date is never null", data["created_date"].notna().all(),
      f"{int(data['created_date'].isna().sum())} nulls")
in_window = all(
    (
        (part["created_date"] >= pd.Timestamp(WINDOWS[label][0]))
        & (part["created_date"] < pd.Timestamp(WINDOWS[label][1]))
    ).all()
    for label, (part, _) in frames.items())
check("every row sits inside its own window", in_window,
      "  ".join(f"{lab}: {part['created_date'].min()} .. {part['created_date'].max()}"
                for lab, (part, _) in frames.items()))
check("row count equals the API's own count(*)",
      all(len(frames[lab][0]) == n for lab, n in api_counts.items()),
      "  ".join(f"{lab.split('_')[0]}: pulled {len(frames[lab][0]):,} / api {n:,}" for lab, n in api_counts.items()))
closed_now = data.assign(closed=data["closed_date"].notna()).groupby("window")["closed"].sum()
check("closed counts match notebook 01",
      int(closed_now.get("event_2024-01-15_22", 0)) == 22770
      and int(closed_now.get("quiet_2024-01-24_30", 0)) == 7309,
      "  ".join(f"{lab.split('_')[0]}: closed {int(closed_now[lab]):,} (nb01 {'event' if lab.startswith('event') else 'quiet'} "
                f"{22770 if lab.startswith('event') else 7309:,})" for lab in closed_now.index))
check("every approved intake column exists", not missing_cols,
      f"{len(missing_cols)} never returned" + (f": {missing_cols}" if missing_cols else ""))

gate = pd.DataFrame(checks)
print(gate.to_string(index=False))
print(f"\n{int((gate['result'] == 'FAIL').sum())} of {len(gate)} checks failed")
print("\nnon-null rate per intake field:")
print(pd.DataFrame({c: [f"{data[c].notna().mean():.1%}", data[c].nunique(dropna=True),
                        f"{data[c].astype(str).str.len().max()}"] for c in INTAKE_FIELDS},
                   index=["filled", "distinct", "max_chars"]).T.to_string())
assert (gate["result"] == "PASS").all(), "stop here: the pull failed an integrity check"

                                  check result                                                                                                                           detail
                   unique_key is unique   PASS                                                                                                   30,126 distinct of 30,126 rows
             created_date is never null   PASS                                                                                                                          0 nulls
   every row sits inside its own window   PASS event_2024-01-15_22: 2024-01-15 00:01:29 .. 2024-01-22 23:59:03  quiet_2024-01-24_30: 2024-01-24 00:03:13 .. 2024-01-30 23:57:14
row count equals the API's own count(*)   PASS                                                               event: pulled 22,811 / api 22,811  quiet: pulled 7,315 / api 7,315
        closed counts match notebook 01   PASS                                                 event: closed 22,770 (nb0

## Duration, and the numbers this notebook must reproduce

`resolution_hours` is **computed here**, never fetched — it is `(closed_date - created_date)`, and the only field
of its kind in the source. If the medians and the quiet p90 do not land on what notebook 01 printed, the data
moved under us and nothing downstream can be trusted.

In [7]:
data["closed_date_dt"] = pd.to_datetime(data["closed_date"])
data["resolution_hours"] = (data["closed_date_dt"] - data["created_date"]).dt.total_seconds() / 3600

NB01_EXPECTED_MEDIAN = {"event_2024-01-15_22": 46.7, "quiet_2024-01-24_30": 28.9}
NB01_EXPECTED_P90 = {"event_2024-01-15_22": 80.4, "quiet_2024-01-24_30": 62.3}
dur = data.dropna(subset=["resolution_hours"]).groupby("window")["resolution_hours"]
report = pd.DataFrame({"closed": dur.size(), "median_h": dur.median(), "p90_h": dur.quantile(.9),
                       "p99_h": dur.quantile(.99), "max_h": dur.max()}).round(1)
report["median_matches_nb01"] = [
    f"{r.median_h:.1f}" == f"{NB01_EXPECTED_MEDIAN[i]:.1f}" for i, r in report.iterrows()]
report["p90_matches_nb01"] = [
    f"{r.p90_h:.1f}" == f"{NB01_EXPECTED_P90[i]:.1f}" for i, r in report.iterrows()]
print(report.to_string())

neg = int((data["resolution_hours"] < 0).sum())
zero = int((data["resolution_hours"] == 0).sum())
print(f"\nnegative durations: {neg}   same-instant closes: {zero}   longest: {data['resolution_hours'].max():,.1f} h")
print(f"open rows with no duration yet: {int(data['resolution_hours'].isna().sum())}")
assert neg == 0, "a closed-before-created ticket would make every duration claim wrong"
assert (report["median_matches_nb01"].all() and report["p90_matches_nb01"].all()), \
    "the window statistics moved away from notebook 01 — re-check the pull before labelling anything"

                     closed  median_h  p90_h  p99_h    max_h  median_matches_nb01  p90_matches_nb01
window                                                                                             
event_2024-01-15_22   22770      46.7   80.4  106.5  12889.8                 True              True
quiet_2024-01-24_30    7309      28.9   62.3   87.6  12715.1                 True              True

negative durations: 0   same-instant closes: 3   longest: 12,889.8 h
open rows with no duration yet: 47


## Threshold and interpretation

Compute the 90th percentile of quiet-window closed-ticket resolution hours in this run. The threshold is a local comparison benchmark, not an SLA or calibrated production cut-off. The quiet-window positive rate is about 10% by construction; the event-window rate is the independent comparison.

In [8]:
quiet = data[(data["window"] == "quiet_2024-01-24_30") & data["resolution_hours"].notna()]
LABEL_THRESHOLD_HOURS = float(quiet["resolution_hours"].quantile(0.9))
QUANTILE_METHOD = "pandas Series.quantile(0.9), linear interpolation"
print(f"quiet closed tickets: {len(quiet):,}")
print(f"derived threshold   : {LABEL_THRESHOLD_HOURS:.6f} h  ->  displayed {LABEL_THRESHOLD_HOURS:.1f} h")
print(f"method              : {QUANTILE_METHOD}")
assert abs(round(LABEL_THRESHOLD_HOURS, 1) - 62.3) < 0.05, \
    f"quiet p90 is {LABEL_THRESHOLD_HOURS:.1f} h, not the 62.3 h notebook 01 reported"


def labelled_view(source):
    """Closed tickets only — the only rows that can carry an outcome. Re-run whenever `data` gains columns."""
    out = source[source["resolution_hours"].notna()].copy()
    out["slow_resolution"] = out["resolution_hours"] > LABEL_THRESHOLD_HOURS
    return out


closed = labelled_view(data)
prev = closed.groupby("window")["slow_resolution"].agg(n_closed="size", n_slow="sum", rate="mean")
prev["rate"] = (prev["rate"] * 100).round(1)
print(f"\n{'window':<22}{'closed':>8}{'slow':>8}{'% slow':>8}")
for lab, r in prev.iterrows():
    print(f"{lab:<22}{int(r.n_closed):>8,}{int(r.n_slow):>8,}{r.rate:>7.1f}%")
print(f"\nquiet rate is 10% by construction; the event rate of {prev.loc['event_2024-01-15_22', 'rate']:.1f}% "
      f"is the measurement that was not fixed in advance.")

quiet closed tickets: 7,309
derived threshold   : 62.349333 h  ->  displayed 62.3 h
method              : pandas Series.quantile(0.9), linear interpolation

window                  closed    slow  % slow
event_2024-01-15_22     22,770   6,526   28.7%
quiet_2024-01-24_30      7,309     731   10.0%

quiet rate is 10% by construction; the event rate of 28.7% is the measurement that was not fixed in advance.


## Open tickets have no outcome yet

An open ticket is not a fast ticket. Its label does not exist, so it stays in the features table and is **absent
from the labels table** — no `False`, no `0`, no imputation. The cell below prints what the rate would look like
if open rows were silently labelled negative, next to the honest number, because that is the failure a reviewer
will ask about.

In [9]:
rows = []
for label, grp in closed.groupby("window"):
    created_n = int((data["window"] == label).sum())
    open_n = created_n - len(grp)
    naive = grp["slow_resolution"].sum() / created_n
    rows.append({"window": label, "created": created_n, "closed": len(grp), "open_no_label": open_n,
                 "honest_rate_%": round(100 * grp["slow_resolution"].mean(), 1),
                 "if_open_were_False_%": round(100 * naive, 1)})
print(pd.DataFrame(rows).to_string(index=False))
total_open = int(data["resolution_hours"].isna().sum())
print(f"\n{total_open} tickets have no label in this snapshot and will simply not appear in the labels file "
      f"(labels rows = {len(closed):,} < features rows = {len(data):,}).")
print("Reconciling with notebook 01: ~99.8% of January intake closes within the window, so the missing "
      "labels are rare here — but the bias is real for recent months, where slow tickets are still open.")

             window  created  closed  open_no_label  honest_rate_%  if_open_were_False_%
event_2024-01-15_22    22811   22770             41           28.7                  28.6
quiet_2024-01-24_30     7315    7309              6           10.0                  10.0

47 tickets have no label in this snapshot and will simply not appear in the labels file (labels rows = 30,079 < features rows = 30,126).
Reconciling with notebook 01: ~99.8% of January intake closes within the window, so the missing labels are rare here — but the bias is real for recent months, where slow tickets are still open.


## Calendar features derived from the intake timestamp

Raw `created_date` is banned as a model input, but *when in the day or week people call* is legitimate intake-time
information and is kept. These features carry the rhythm without carrying the calendar, so a model cannot
memorise the event week by date.

No train/test split is created here. Choosing one now would fix the evaluation design before the label
distribution above is known; it belongs to the notebook that runs the evaluation.

In [10]:
data["created_hour"] = data["created_date"].dt.hour
data["created_day_of_week"] = data["created_date"].dt.day_name()
data["created_is_weekend"] = data["created_date"].dt.dayofweek >= 5
CAL_FEATURES = ["created_hour", "created_day_of_week", "created_is_weekend"]

# `closed` was taken from `data` in the threshold cell, before these columns existed.
closed = labelled_view(data)
print(f"\nlabelled view rebuilt: {len(closed):,} closed rows x {len(closed.columns)} columns, "
      f"now including {CAL_FEATURES}")

for f in CAL_FEATURES:
    vc = data[f].value_counts()
    print(f"{f:<22} {data[f].nunique():>3} distinct   top: {vc.index[0]} = {vc.iloc[0]:,} "
          f"({vc.iloc[0] / len(data):.1%})")
print(f"\nweekend intake share: {data['created_is_weekend'].mean():.1%}   "
      f"(notebook 01 showed quiet days are not slack days, so this is demand, not staffing)")


labelled view rebuilt: 30,079 closed rows x 18 columns, now including ['created_hour', 'created_day_of_week', 'created_is_weekend']
created_hour            24 distinct   top: 9 = 2,106 (7.0%)
created_day_of_week      7 distinct   top: Monday = 6,154 (20.4%)
created_is_weekend       2 distinct   top: False = 22,408 (74.4%)

weekend intake share: 25.6%   (notebook 01 showed quiet days are not slack days, so this is demand, not staffing)


## Feature contract

Assertions check the allowlist against the live source schema. `MODEL_INPUTS` contains approved intake and calendar features. `FEATURE_COLS` additionally includes `unique_key`, `created_date`, and `window` for audit and chronological splitting; those three fields are never model inputs.

In [11]:
MODEL_INPUTS = INTAKE_FIELDS + CAL_FEATURES
FEATURE_COLS = ["unique_key", "created_date", "window"] + MODEL_INPUTS
LABEL_COLS = ["unique_key", "slow_resolution", "label_threshold_hours"]

guards = []


def guard(name, ok, detail):
    guards.append({"assertion": name, "result": "PASS" if ok else "FAIL", "detail": detail})


guard("file columns are only intake + audit + calendar",
      set(FEATURE_COLS) <= set(INTAKE_FIELDS) | set(AUDIT_ONLY) | set(CAL_FEATURES),
      f"{len(FEATURE_COLS)} columns declared")
guard("no forbidden field is in the features file",
      not set(FEATURE_COLS) & set(FORBIDDEN_FIELDS),
      f"{len(FORBIDDEN_FIELDS)} forbidden names checked against {len(FEATURE_COLS)} columns")
guard("no closure or label column is in the features file",
      not set(FEATURE_COLS) & {"closed_date", "closed_date_dt", "status", "resolution_hours", "slow_resolution"},
      "closed_date, status, resolution_hours, slow_resolution all excluded")
guard("model inputs exclude audit-only identifiers and dates",
      not set(MODEL_INPUTS) & {"unique_key", "created_date", "window"},
      "created_date and window are audit/time-split only")
guard("model inputs exist in the source or are derived here",
      set(MODEL_INPUTS) <= ALL_SOURCE | set(CAL_FEATURES),
      f"{len([f for f in MODEL_INPUTS if f not in ALL_SOURCE])} derived in this notebook")
guard("labels columns carry no closure or duration field",
      not set(LABEL_COLS) & (set(FORBIDDEN_FIELDS) | {"resolution_hours", "closed_date_dt"}),
      f"{LABEL_COLS} — the yardstick travels with the label, nothing else does")
forbidden_cols = [c for c in data.columns if c in FORBIDDEN_FIELDS]
print(pd.DataFrame(guards).to_string(index=False))
print(f"\nMODEL_INPUTS ({len(MODEL_INPUTS)}): {MODEL_INPUTS}")
print(f"carried for audit only: {['unique_key', 'created_date', 'window']}")
print(f"columns present in the frame but never written to either output: {forbidden_cols}")
assert all(g["result"] == "PASS" for g in guards), "a leakage rule failed — nothing may be written"

                                            assertion result                                                                                                               detail
      file columns are only intake + audit + calendar   PASS                                                                                                  13 columns declared
           no forbidden field is in the features file   PASS                                                                        39 forbidden names checked against 13 columns
   no closure or label column is in the features file   PASS                                                  closed_date, status, resolution_hours, slow_resolution all excluded
model inputs exclude audit-only identifiers and dates   PASS                                                                    created_date and window are audit/time-split only
 model inputs exist in the source or are derived here   PASS                                                  

## Exploratory signal table

One row per intake category, on closed tickets: how many, the median and p90 duration, and the share flagged
slow at the derived threshold. `created_hour` is read in six-hour blocks so 24 levels do not hide the pattern.

**This is exploratory evidence, not an estimate of accuracy.** It shows whether an intake field varies enough to
be worth carrying forward; it says nothing about how well any model would predict, and no such claim is made from
it. A field with one value cannot inform anything and is reported as such rather than dropped quietly.

In [12]:
sig = closed.assign(created_hour_bin=pd.cut(closed["created_hour"], [-1, 5, 11, 17, 23],
                                            labels=["00-05", "06-11", "12-17", "18-23"]))
summary = []
for f in MODEL_INPUTS:
    col = "created_hour_bin" if f == "created_hour" else f
    grp = sig.groupby(col, dropna=False)
    table = pd.DataFrame({"n": grp.size(), "median_h": grp["resolution_hours"].median().round(1),
                          "p90_h": grp["resolution_hours"].quantile(.9).round(1),
                          "slow_%": (100 * grp["slow_resolution"].mean()).round(1)})
    spread = float(table["slow_%"].max() - table["slow_%"].min())
    levels = len(table)
    table_ = table.sort_values("n", ascending=False)
    summary.append({"feature": f, "levels": levels, "distinct_in_closed": sig[col].nunique(dropna=True),
                    "slow_spread_pp": round(spread, 1),
                    "worst_level": str(table_["slow_%"].idxmax()), "worst_slow_%": table_["slow_%"].max(),
                    "best_level": str(table_["slow_%"].idxmin()), "best_slow_%": table_["slow_%"].min()})
    if levels <= 6:
        print(f"\n{f}")
        print(table_.to_string())

s = pd.DataFrame(summary).sort_values("slow_spread_pp", ascending=False)
print(f"\n{'feature':<24}{'levels':>7}{'spread_pp':>11}{'worst level':>26}{'slow_%':>8}"
      f"{'best level':>24}{'slow_%':>8}")
for _, r in s.iterrows():
    print(f"{r['feature']:<24}{r['levels']:>7}{r['slow_spread_pp']:>11.1f}"
          f"{r['worst_level']:>26}{r['worst_slow_%']:>8.1f}{r['best_level']:>24}{r['best_slow_%']:>8.1f}")
flat = s[s["levels"] <= 1]["feature"].tolist()
print(f"\nconstant within these two windows, so no signal available: {flat}")
print("exploratory only — this table does not estimate how well anything could be predicted.")


complaint_type
                    n  median_h  p90_h  slow_%
complaint_type                                
HEAT/HOT WATER  30079      41.7   77.8    24.1

descriptor
                     n  median_h  p90_h  slow_%
descriptor                                     
ENTIRE BUILDING  18774      39.3   77.2    22.8
APARTMENT ONLY   11305      44.9   79.0    26.3

descriptor_2
                              n  median_h  p90_h  slow_%
descriptor_2                                            
NO HEAT                   19222      43.9   79.0    26.3
NO HEAT AND NO HOT WATER   8709      37.6   76.0    20.4
NO HOT WATER               2148      35.9   75.8    19.6

location_type
                          n  median_h  p90_h  slow_%
location_type                                       
RESIDENTIAL BUILDING  30079      41.7   77.8    24.1

borough
                   n  median_h  p90_h  slow_%
borough                                      
BRONX          10651      31.2   66.9    12.6
BROOKLYN        782

## Write the two tables

Two files, named for what they actually cover — `jan2024` and `event_quiet`, so nobody later mistakes them for a
month-wide or city-wide table:

- **features**: one row per *created* ticket, open ones included. Intake fields, the three calendar features, and
  `unique_key` / `created_date` / `window` for audit. No closure field.
- **labels**: one row per *closed* ticket, and exactly three columns — `unique_key`, `slow_resolution`,
  `label_threshold_hours`. The threshold travels inside the file so a label can never be reinterpreted without
  its own yardstick.

`closed_date`, `status` and `resolution_hours` are used in memory to build the labels and are never written to
either file.

In [13]:
def clean(v):
    if v is pd.NaT:
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(v, pd.Timestamp):
        return v.isoformat()
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, (np.bool_,)):
        return bool(v)
    return v


features = data[FEATURE_COLS].copy()
assert not set(features.columns) & set(FORBIDDEN_FIELDS)
feature_recs = [{k: clean(v) for k, v in r.items()} for r in features.to_dict(orient="records")]
label_recs = [{"unique_key": clean(k), "slow_resolution": clean(b),
               "label_threshold_hours": LABEL_THRESHOLD_HOURS}
              for k, b in zip(closed["unique_key"], closed["slow_resolution"])]

features_path = out_dir / "heat_jan2024_event_quiet_features_at_intake.json"
labels_path = out_dir / "heat_jan2024_event_quiet_slow_resolution_labels.json"
for path, recs in ((features_path, feature_recs), (labels_path, label_recs)):
    body = json.dumps(recs, indent=2)
    path.write_text(body, encoding="utf-8")
    print(f"{path.name:<52} {len(recs):>6,} rows   {path.stat().st_size:>9,} bytes   "
          f"sha256 {hashlib.sha256(body.encode('utf-8')).hexdigest()[:16]}")
print(f"\nfeatures columns: {list(feature_recs[0])}")
print(f"labels   columns: {list(label_recs[0])}")

heat_jan2024_event_quiet_features_at_intake.json     30,126 rows   14,221,430 bytes   sha256 a5af29b602258c71
heat_jan2024_event_quiet_slow_resolution_labels.json 30,079 rows   3,632,305 bytes   sha256 ba17f83f0a6f11cc

features columns: ['unique_key', 'created_date', 'window', 'complaint_type', 'descriptor', 'descriptor_2', 'location_type', 'borough', 'open_data_channel_type', 'agency', 'created_hour', 'created_day_of_week', 'created_is_weekend']
labels   columns: ['unique_key', 'slow_resolution', 'label_threshold_hours']


## Prove the files on disk match what we wrote

Both files are re-read from disk as a fresh reader would and re-checked, so the claim is about the artefact and
not about a dataframe that only exists while the kernel is alive.

In [14]:
feat_disk = pd.read_json(features_path, dtype={"unique_key": "int64"})
lab_disk = pd.read_json(labels_path, dtype={"unique_key": "int64"})
round_trip = []


def rt(name, ok, detail):
    round_trip.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})


rt("features row count", len(feat_disk) == len(data), f"{len(feat_disk):,} on disk / {len(data):,} in memory")
rt("labels row count", len(lab_disk) == len(closed), f"{len(lab_disk):,} on disk / {len(closed):,} closed")
rt("unique_key unique in both", feat_disk["unique_key"].is_unique and lab_disk["unique_key"].is_unique, "no duplicates")
rt("labels are a subset of features by key", set(lab_disk["unique_key"]) <= set(feat_disk["unique_key"]),
   f"{len(set(feat_disk['unique_key']) - set(lab_disk['unique_key']))} features rows have no label")
rt("missing labels == open tickets",
   len(feat_disk) - len(lab_disk) == total_open, f"{len(feat_disk) - len(lab_disk)} vs {total_open} open")
rt("no forbidden column on disk", not set(feat_disk.columns) & set(FORBIDDEN_FIELDS),
   f"{len(feat_disk.columns)} feature columns / {len(lab_disk.columns)} label columns")
rt("labels file holds exactly the three agreed columns", list(lab_disk.columns) == LABEL_COLS,
   f"{list(lab_disk.columns)}")
rt("threshold stored exactly", lab_disk["label_threshold_hours"].nunique() == 1
   and abs(lab_disk["label_threshold_hours"].iloc[0] - LABEL_THRESHOLD_HOURS) < 1e-12,
   f"{lab_disk['label_threshold_hours'].iloc[0]:.6f} h in every row")
recomputed = round(100 * lab_disk["slow_resolution"].mean(), 1)
rt("prevalence recomputed from disk", recomputed == round(100 * closed["slow_resolution"].mean(), 1),
   f"{recomputed}% slow overall, recomputed from the file")

print(pd.DataFrame(round_trip).to_string(index=False))
back = lab_disk.merge(feat_disk[["unique_key", "window"]], on="unique_key")
rates = (100 * back.groupby("window")["slow_resolution"].mean()).round(1)
print("\n" + str(int((pd.DataFrame(round_trip)["result"] == "FAIL").sum()))
      + f" of {len(round_trip)} failed")
print("slow rate per window, recomputed from the files on disk: "
      + "  ".join(f"{lab.split('_')[0]} {r}%" for lab, r in rates.items()))
assert all(r["result"] == "PASS" for r in round_trip), "the files on disk disagree with the run"

                                             check result                                                     detail
                                features row count   PASS                          30,126 on disk / 30,126 in memory
                                  labels row count   PASS                             30,079 on disk / 30,079 closed
                         unique_key unique in both   PASS                                              no duplicates
            labels are a subset of features by key   PASS                             47 features rows have no label
                    missing labels == open tickets   PASS                                              47 vs 47 open
                       no forbidden column on disk   PASS                       13 feature columns / 3 label columns
labels file holds exactly the three agreed columns   PASS ['unique_key', 'slow_resolution', 'label_threshold_hours']
                          threshold stored exactly   PASS       

## Provenance: sidecars

One `.meta.json` beside each file, matching the convention already used across `data/raw`. Each records the
parameters that were actually sent, the exact threshold with the method that produced it, the two field lists
that define the contract, and the limits that a reader must know before quoting a number from these files.

In [15]:
built_at = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
common = {
    "built_at": built_at,
    "source_url": url,
    "metadata_url": meta_url,
    "complaint_filter": HEAT,
    "windows": {lab: {"created_from": d0, "created_before": d1,
                      "rows_pulled": len(frames[lab][0]), "count_from_api": api_counts[lab]}
                for lab, (d0, d1) in WINDOWS.items()},
    "select": SELECT,
    "paging": PULL_PARAMS,
    "requested_includes_closure_fields": ["closed_date", "status"],
    "closure_fields_written_to_output": False,
    "intake_fields": INTAKE_FIELDS,
    "derived_features": CAL_FEATURES,
    "audit_only_fields": AUDIT_ONLY,
    "model_inputs": MODEL_INPUTS,
    "forbidden_fields": FORBIDDEN_FIELDS,
    "deliberately_excluded": NOTES,
    "review_flags": REVIEW_FLAGS,
    "http_requests_used": requests_used + len(api_counts),
}
label_common = {
    **common,
    "label_threshold_hours": LABEL_THRESHOLD_HOURS,
    "label_threshold_displayed_h": round(LABEL_THRESHOLD_HOURS, 1),
    "threshold_source": "p90 of the quiet window's closed tickets pulled in this run, not a published standard",
    "threshold_method": QUANTILE_METHOD,
    "threshold_fitted_on": {"window": "quiet_2024-01-24_30", "n_closed": len(quiet)},
    "drift_check_vs_notebook_01": "abs(round(p90,1) - 62.3) < 0.05",
    "label_rule": "slow_resolution = resolution_hours > label_threshold_hours, closed tickets only",
    "open_tickets": {"count": total_open, "label": "none — absent from this file, never labelled False"},
    "known_limits": [
        "threshold is fitted on the quiet window and then used to label it, so it demonstrates separation, not calibration",
        "quiet-window positive rate is ~10% by construction; only the event-window rate is a free measurement",
        "closed tickets only carry labels, so slow-but-still-open tickets are missing by design",
        "two windows of one month of one complaint type: a case study, not a citywide model",
        "intake signal table is exploratory evidence, not an accuracy estimate",
        "borough is coarse service geography and still needs a fairness/proxy review before it is served",
    ],
}
for path, extra in ((features_path, {"rows": len(feature_recs), "columns": list(feature_recs[0])}),
                    (labels_path, {"rows": len(label_recs), "columns": LABEL_COLS,
                                   "slow_overall_pct": round(100 * closed["slow_resolution"].mean(), 1),
                                   "slow_by_window_pct": {lab.split("_")[0]: float(r)
                                                          for lab, r in prev["rate"].items()}})):
    body = {**common, **({} if path is features_path else label_common), **extra}
    body["artifact_sha256"] = hashlib.sha256(path.read_bytes()).hexdigest()
    body["artifact_bytes"] = path.stat().st_size
    if path is features_path:
        body["open_tickets_included"] = True
        body["known_limits"] = [
            "one row per created ticket, including tickets with no label yet",
            "created_date and window travel here for audit and time-splitting only; they are not model inputs",
            "borough is coarse service geography and still needs a fairness/proxy review before it is served",
        ]
    side = path.with_name(path.stem + ".meta.json")
    body_text = json.dumps(body, indent=2)
    side.write_text(body_text, encoding="utf-8")
    print(f"{side.name:<56} {len(body):>3} keys   {side.stat().st_size:>6,} bytes   "
          f"sha256 {hashlib.sha256(body_text.encode('utf-8')).hexdigest()[:16]}")
print(f"\nthreshold written to disk: {LABEL_THRESHOLD_HOURS:.6f} h (displayed {LABEL_THRESHOLD_HOURS:.1f} h)")

heat_jan2024_event_quiet_features_at_intake.meta.json     23 keys    3,977 bytes   sha256 57d09e7de026ab36
heat_jan2024_event_quiet_slow_resolution_labels.meta.json  32 keys    4,810 bytes   sha256 387bcd4ff00a9d1e

threshold written to disk: 62.349333 h (displayed 62.3 h)


## Interpretation and limits

The label marks closed requests whose recorded resolution time exceeds the quiet-window p90. The event/quiet comparison describes two windows in January 2024; it does not establish a City service standard, calibration, causation, or predictive accuracy. The signal table is exploratory. Open requests receive no label.

Downstream models may use `MODEL_INPUTS` only. `unique_key`, `created_date`, and `window` are for provenance and chronological splitting, not predictors.